In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

# Set style for plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

# Load the California Housing dataset
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = pd.Series(california.target, name='MedHouseVal')

# Create a DataFrame for easier manipulation
df = X.copy()
df['MedHouseVal'] = y

# Select features based on correlation
selected_features = ['MedInc', 'AveRooms', 'Latitude']
X_selected = X[selected_features]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear Regression Model
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
y_pred_linear = linear_model.predict(X_test_scaled)
mse_linear = mean_squared_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mse_linear)
r2_linear = r2_score(y_test, y_pred_linear)

# Polynomial Regression Model (Degree 2)
poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])
poly_model.fit(X_train, y_train)
y_pred_poly = poly_model.predict(X_test)
mse_poly = mean_squared_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mse_poly)
r2_poly = r2_score(y_test, y_pred_poly)

# Polynomial Regression Model (Degree 3)
poly_model_3 = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])
poly_model_3.fit(X_train, y_train)
y_pred_poly_3 = poly_model_3.predict(X_test)
r2_poly_3 = r2_score(y_test, y_pred_poly_3)

# Results Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Actual vs Predicted - Linear Regression
axes[0, 0].scatter(y_test, y_pred_linear, alpha=0.6, color='blue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Values')
axes[0, 0].set_ylabel('Predicted Values')
axes[0, 0].set_title('Linear Regression: Actual vs Predicted')
axes[0, 0].text(0.05, 0.95, f'R² = {r2_linear:.3f}', transform=axes[0, 0].transAxes,
         fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Actual vs Predicted - Polynomial Regression
axes[0, 1].scatter(y_test, y_pred_poly, alpha=0.6, color='green')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Values')
axes[0, 1].set_ylabel('Predicted Values')
axes[0, 1].set_title('Polynomial Regression: Actual vs Predicted')
axes[0, 1].text(0.05, 0.95, f'R² = {r2_poly:.3f}', transform=axes[0, 1].transAxes,
         fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Residual plots - Linear Regression
residuals_linear = y_test - y_pred_linear
axes[1, 0].scatter(y_pred_linear, residuals_linear, alpha=0.6, color='blue')
axes[1, 0].axhline(y=0, color='red', linestyle='--')
axes[1, 0].set_xlabel('Predicted Values')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Linear Regression: Residual Plot')

# Residual plots - Polynomial Regression
residuals_poly = y_test - y_pred_poly
axes[1, 1].scatter(y_pred_poly, residuals_poly, alpha=0.6, color='green')
axes[1, 1].axhline(y=0, color='red', linestyle='--')
axes[1, 1].set_xlabel('Predicted Values')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Polynomial Regression: Residual Plot')

plt.tight_layout()
plt.show()

# Print results
print("=" * 60)
print("REGRESSION MODEL COMPARISON")
print("=" * 60)
print(f"Linear Regression R²: {r2_linear:.4f}")
print(f"Polynomial Regression (Degree 2) R²: {r2_poly:.4f}")
print(f"Polynomial Regression (Degree 3) R²: {r2_poly_3:.4f}")
print(f"Improvement (Linear to Poly2): {((r2_poly - r2_linear)/r2_linear)*100:.1f}%")
print(f"Improvement (Poly2 to Poly3): {((r2_poly_3 - r2_poly)/r2_poly)*100:.1f}%")

# Model coefficients
coefficients = pd.DataFrame({
    'Feature': ['Intercept'] + selected_features,
    'Coefficient': [linear_model.intercept_] + list(linear_model.coef_)
})
print("\nLinear Regression Coefficients:")
print(coefficients)